# 08 · Race simulator (dry races)

Simulates a race thousands of times with any field you choose: drivers from any season they raced (2018–2025), in any team-season's car, at any circuit.

**Each simulated race**
1. Draws one sample from the driver/car model (notebook 07), so uncertainty about skill and cars carries into the results.
2. **Qualifying:** one-lap pace = driver skill + car pace at this circuit, plus qualifying noise. The order sets the grid. The noise is heavy-tailed only on the slow side (crashes, traffic), because real outliers are almost all slow; a symmetric tail gave midfield cars about 21 poles in 152 dry races, against 0 in reality.
3. **Retirements:** each car retires at its team-season's DNF rate, adjusted for the circuit's DNF rate (dry races; shrunk toward the average when data is thin).
4. **Race:** starting from the grid, a car passes the car ahead only if it's faster by more than the circuit's **overtaking threshold**. Thresholds are calibrated so simulated position changes match each circuit's real dry races.
5. **Points:** 25-18-15-12-10-8-6-4-2-1 (no fastest-lap point, as in 2025).

**Caveat:** for real driver–car pairings, skill + car is well measured. For **swapped** entries (a driver in a car they never drove), results depend on how the model split skill from car, which teammate comparisons only partly pin down; the intervals don't capture all of that. Swapped entries are flagged.

Run notebooks 03, 05 (with circuit information) and 07 first.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Install PyMC and the file format for saved fits (skipped if already installed)
import importlib.util
missing = [p for p in ("pymc", "h5netcdf", "h5py") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [ ]:
# @title Load the fit, DNF rates and overtaking thresholds
RECALIBRATE = False  # @param {type:"boolean"}

import pandas as pd
import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest_results import load_results
from f1sim.ingest_tracing import load_race_ti
from f1sim.race_pace import clean_dry_laps
from f1sim import skill_model as sm
from f1sim import simulator as sim

paths = get_paths()
results = load_results(paths)
cal = sim.calendar(results)     # fails with instructions if circuit info is missing
idata, data = sm.load_fit(paths)
post = sim.load_posterior(idata, data)

# Dry races = races the model used (no wet tyres, at least some clean laps)
dry_keys = set(map(tuple, clean_dry_laps(load_race_ti("laps", paths))[["Year", "Round"]]
                   .drop_duplicates().to_numpy()))
dnf = sim.dnf_rates(results, dry_keys)

thr_file = paths.models_dir / "overtaking_thresholds.parquet"
if RECALIBRATE or not thr_file.exists():
    print("Calibrating overtaking thresholds (a few seconds)...")
    thresholds = sim.calibrate_overtaking(results, post, dry_keys)
    thresholds.to_parquet(thr_file, index=False)
else:
    thresholds = pd.read_parquet(thr_file)
print(f"{post.n_draws} posterior draws | {len(dry_keys)} dry races | "
      f"overall DNF rate {100 * dnf.overall:.1f}%")
thresholds

In [ ]:
# @title Choose the race and the field
BASE_YEAR = 2025  # @param {type:"integer"}
BASE_ROUND = 24  # @param {type:"integer"}
CIRCUIT = ""  # @param {type:"string"}
DRIVER_SWAPS = ""  # @param {type:"string"}
CAR_CHANGES = ""  # @param {type:"string"}
DNFS = True  # @param {type:"boolean"}

# Field: the real field of BASE_YEAR round BASE_ROUND, then your changes.
#   CIRCUIT       blank = that race's circuit, or a CircuitId from the table above (e.g. monaco)
#   DRIVER_SWAPS  OLD=NEW or OLD=NEW@SEASON, ';'-separated. The new driver takes the old one's car.
#                 e.g.  TSU=HAM@2019; LAW=VET@2018      (no @SEASON = their latest season)
#   CAR_CHANGES   DRIVER=TEAM or DRIVER=TEAM@SEASON, e.g.  VER=mclaren; HAM=mercedes@2020

field = sim.field_from_race(results, BASE_YEAR, BASE_ROUND)
if DRIVER_SWAPS.strip():
    field = sim.apply_driver_swaps(field, results, post, DRIVER_SWAPS)
if CAR_CHANGES.strip():
    field = sim.apply_car_changes(field, results, CAR_CHANGES)

base = cal[(cal["Year"] == BASE_YEAR) & (cal["Round"] == BASE_ROUND)].iloc[0]
circuit = CIRCUIT.strip() or base["CircuitId"]
if circuit not in set(cal["CircuitId"]):
    raise KeyError(f"Unknown circuit '{circuit}'. Options: {sorted(cal['CircuitId'].unique())}")
row = thresholds[thresholds["CircuitId"] == circuit]
threshold = float(row["Threshold"].iloc[0]) if len(row) else float(thresholds["Threshold"].median())
entries = sim.describe_field(field, results, post, circuit)

print(f"Circuit: {cal.loc[cal['CircuitId'] == circuit, 'CircuitName'].iloc[0]} | "
      f"overtaking threshold {threshold:.2f}% of a lap"
      + ("" if len(row) else " (no dry races here: median used)"))
per_team = entries.groupby(["TeamId", "CarYear"]).size()
if (per_team != 2).any():
    print("Note: some teams don't have exactly two cars:", per_team[per_team != 2].to_dict())
entries.drop(columns=["_skill_cell", "_car_cells", "DriverId", "TeamId"])

In [ ]:
# @title Simulate
N_SIMS = 5000  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}

res = sim.simulate(entries, post, circuit, threshold, dnf if DNFS else None,
                   n_sims=N_SIMS, seed=SEED)
summary = res.summary()
if summary["Swapped"].any():
    print("Swapped entries (driver in a car they didn't race that season): "
          + ", ".join(summary.loc[summary["Swapped"], "Driver"])
          + ". Treat their results as more uncertain than the intervals show.")
summary

In [ ]:
# @title Win, podium and points chances
long = summary.melt(id_vars=["Driver", "Team"], value_vars=["Win%", "Podium%", "Points%"],
                    var_name="Outcome", value_name="Percent")
fig = px.bar(long, x="Driver", y="Percent", color="Outcome", barmode="group",
             hover_data=["Team"], category_orders={"Driver": summary["Driver"].tolist()},
             title=f"Simulated chances over {N_SIMS:,} races")
fig.show()

In [ ]:
# @title Finishing position probabilities
probs = res.position_probabilities().loc[summary["Driver"]]
fig = px.imshow(probs, text_auto=".0f", aspect="auto", color_continuous_scale="Blues",
                labels=dict(x="Finishing position", y="Driver", color="% of races"),
                title="Chance of each finishing position (%)")
fig.update_layout(height=700)
fig.show()

In [ ]:
# @title Expected points per team
res.team_points()